In [1]:
!pip install transformers[torch]==4.57.6 evaluate ipywidgets requests

In [2]:
from datasets import load_dataset, load_from_disk, Value
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, RobertaTokenizerFast, RobertaForSequenceClassification, DataCollatorWithPadding
from datasets import ClassLabel
import torch
import numpy as np
import evaluate
import requests

In [3]:
assert torch.cuda.is_available()
assert torch.cuda.is_bf16_supported()

In [4]:
SOURCE_DATA_FILE = "data/fr-en/fr-machine-translation-10k.parquet"
TOKENIZED_DATA_FILE = "data/fr-en/fr-machine-translation-10k_tokenized.parquet"
OUTPUT_DIR = "./data/fr-en/fr-machine-translation-10k-out"

In [5]:
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")
ds = load_dataset("parquet", data_files=SOURCE_DATA_FILE)["train"]
    
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)


print(ds)
    
tokenized_ds = (ds.map(tokenize_batch, batched=True).rename_column("label", "labels"))
tokenized_ds.to_parquet(TOKENIZED_DATA_FILE)

Dataset({
    features: ['text', 'label'],
    num_rows: 20000
})


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

5577273

In [7]:
tokenized_ds = tokenized_ds.cast_column(
    "labels", 
    ClassLabel(num_classes=2, names=["human", "machine"])
)

tokenized_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

split = tokenized_ds.train_test_split(
    test_size=0.1, 
    seed=1865, 
    stratify_by_column="labels"
)

train_ds = split["train"]
test_ds = split["test"]

# check distribution
print("Train set distribution:", torch.bincount(torch.tensor(train_ds["labels"])))
print("Test set distribution:", torch.bincount(torch.tensor(test_ds["labels"])))

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_ds_full = train_ds

Train set distribution: tensor([9000, 9000])
Test set distribution: tensor([1000, 1000])


In [8]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
    problem_type="single_label_classification", 
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    bf16=True,
    dataloader_num_workers=8,
    dataloader_pin_memory=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    weight_decay=0.001,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
500,0.624400,0.644024,0.641000
1000,0.593000,0.583694,0.674000
1500,0.530300,0.608926,0.668500
2000,0.519700,0.576932,0.682500
2500,0.432600,0.635524,0.698500
3000,0.452000,0.587334,0.704000


TrainOutput(global_step=3375, training_loss=0.5283716170699508, metrics={'train_runtime': 239.4907, 'train_samples_per_second': 225.479, 'train_steps_per_second': 14.092, 'total_flos': 1353509046535680.0, 'train_loss': 0.5283716170699508, 'epoch': 3.0})

In [10]:
# from transformers.trainer_utils import get_last_checkpoint
# last_checkpoint = get_last_checkpoint("./roberta-year-regressor-day")

In [11]:
indices = np.random.choice(len(test_ds), 30, replace=False)

small_ds = test_ds.select(indices)

pred_output = trainer.predict(small_ds)

preds = pred_output.predictions.squeeze()
labels = pred_output.label_ids

for i, idx in enumerate(indices):
    text = tokenizer.decode(test_ds[idx]["input_ids"], skip_special_tokens=True)
    print(f"Text: {text}")
    
    predicted_class = np.argmax(preds[i])
    actual_class = labels[i]
    
    print(f"Prediction: {predicted_class} | Label: {actual_class}")
    print("=" * 50)

Text: (Parliament rejected the proposal) President.
Prediction: 0 | Label: 0
Text: There are always people who can refuse everything and on the other hand we, who stand here, begging for some extra money for culture.
Prediction: 1 | Label: 1
Text: We wish to ensure that Article 1 of the directive promotes the sustainable, efficient, fair and considerate use of water.
Prediction: 1 | Label: 0
Text: Question No 31 by (H-0795/99):
Prediction: 0 | Label: 0
Text: Despite the many other concerns we may have, ESB has left us a terrible legacy that needs to be addressed and addressed.
Prediction: 1 | Label: 1
Text: Some proposed amendments which we support should strengthen the European Parliament' s negotiating position in the forthcoming conciliation with the Council of Ministers.
Prediction: 0 | Label: 0
Text: I am sure that the Commission is not alone to blame, but I think we must act more quickly to achieve harmonization in this area as well.
Prediction: 1 | Label: 1
Text: The van Hulten 

In [12]:
eval_loss = None
train_loss = None
eval_rmse = None
for log in trainer.state.log_history:
    train_loss = log.get("train_loss") or train_loss
    eval_loss = log.get("eval_loss") or eval_loss
    eval_rmse = log.get("eval_rmse") or eval_rmse

In [14]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# predictions on test dataset
print("Running predictions on the full test dataset...")
full_pred_output = trainer.predict(test_ds)

# extract predictions and true labels
logits = full_pred_output.predictions
predicted_classes = np.argmax(logits, axis=-1)
true_classes = full_pred_output.label_ids

# classification metrics
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(true_classes, predicted_classes, target_names=["Human (0)", "Machine (1)"]))

print("\n" + "="*50)
print("CONFUSION MATRIX")
print("="*50)
cm = confusion_matrix(true_classes, predicted_classes)
print(f"True Negatives (Human correctly identified): {cm[0][0]}")
print(f"False Positives (Human incorrectly flagged as Machine): {cm[0][1]}")
print(f"False Negatives (Machine missed as Human): {cm[1][0]}")
print(f"True Positives (Machine correctly identified): {cm[1][1]}")

# error analysis
print("\n" + "="*50)
print("ERROR ANALYSIS: SAMPLES OF MISCLASSIFIED TEXT")
print("="*50)

# filter for rows where the model made a mistake
df = pd.DataFrame({
    "text": [tokenizer.decode(ids, skip_special_tokens=True) for ids in test_ds["input_ids"]],
    "true_label": true_classes,
    "prediction": predicted_classes
})

errors_df = df[df["true_label"] != df["prediction"]]

# find false positives (human text predicted as machine)
false_positives = errors_df[(errors_df["true_label"] == 0) & (errors_df["prediction"] == 1)]
print(f"\nTotal False Positives: {len(false_positives)}")
if not false_positives.empty:
    print("Sample False Positive (Human text the model thought was Machine):")
    # Grab one random example
    print(f'"{false_positives.sample(1)["text"].values[0]}"')

# find false negatives (machine text predicted as human)
false_negatives = errors_df[(errors_df["true_label"] == 1) & (errors_df["prediction"] == 0)]
print(f"\nTotal False Negatives: {len(false_negatives)}")
if not false_negatives.empty:
    print("Sample False Negative (Machine text the model thought was Human):")
    print(f'"{false_negatives.sample(1)["text"].values[0]}"')

Running predictions on the full test dataset...



CLASSIFICATION REPORT
              precision    recall  f1-score   support

   Human (0)       0.80      0.55      0.65      1000
 Machine (1)       0.65      0.86      0.74      1000

    accuracy                           0.70      2000
   macro avg       0.73      0.70      0.70      2000
weighted avg       0.73      0.70      0.70      2000


CONFUSION MATRIX
True Negatives (Human correctly identified): 545
False Positives (Human incorrectly flagged as Machine): 455
False Negatives (Machine missed as Human): 137
True Positives (Machine correctly identified): 863

ERROR ANALYSIS: SAMPLES OF MISCLASSIFIED TEXT

Total False Positives: 455
Sample False Positive (Human text the model thought was Machine):
"The Commission has therefore adopted the strategic programme for 2000-2005, a programme that was immediately sent to the European Parliament and with which you are already familiar, so I will not go into detail about it now."

Total False Negatives: 137
Sample False Negative (Machin